In [1]:
%pip install phonenumbers --quiet

Note: you may need to restart the kernel to use updated packages.


In [2]:
import re
import phonenumbers

Xử lý cho czechia

In [3]:
import pandas as pd

# Đường dẫn tới 2 file
gd_path = r"C:\Users\Nhung\Downloads\We_Love_Pho\sample structure.csv"
checkpoint_path = r'C:\Users\Nhung\Downloads\We_Love_Pho\raw_country_extracted_6-6\France.csv'

# 1. Đọc dữ liệu
df_checkpoint = pd.read_csv(checkpoint_path)
df_gd = pd.read_csv(gd_path)

In [4]:
df_checkpoint.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7245 entries, 0 to 7244
Data columns (total 14 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   post_title          7245 non-null   object 
 1   address             7245 non-null   object 
 2   latitude            7245 non-null   float64
 3   longitude           7245 non-null   float64
 4   phone               7000 non-null   object 
 5   website             4302 non-null   object 
 6   facebook            0 non-null      float64
 7   instagram           0 non-null      float64
 8   twitter             0 non-null      float64
 9   post_content        7245 non-null   object 
 10  google_maps_link    7245 non-null   object 
 11  google_profile      7245 non-null   object 
 12  google_review_link  7245 non-null   object 
 13  country             7245 non-null   object 
dtypes: float64(5), object(9)
memory usage: 792.6+ KB


In [5]:
# --- Kiểm tra định dạng số điện thoại trước khi chuẩn hóa ---
phones = df_checkpoint["phone"].astype(str).str.strip()

# Loại bỏ các nan
valid_phones = phones[~phones.str.lower().isin(["nan", "none", ""]) & (phones != "")]

# Kiểm tra định dạng số điện thoại
# Có bắt đầu bằng '00'?
count_00 = valid_phones.str.startswith("00").sum()
# Có bắt đầu bằng '+' ?
count_plus = valid_phones.str.startswith("+").sum()
# Có dấu '-'?
count_dash = valid_phones.str.contains("-", regex=False).sum()
# Có dấu cách ?
count_space = valid_phones.str.contains(" ", regex=False).sum()

# Kiểm tra range số điện thoại
phones_cleaned = valid_phones.str.replace(r"[-\s]", "", regex=True)
lengths = phones_cleaned.str.len()
min_len = lengths.min()
max_len = lengths.max()

# --- In kết quả ---
print(f"Số bắt đầu bằng '00': {count_00}")
print(f"Số bắt đầu bằng '+': {count_plus}")
print(f"Số chứa dấu '-': {count_dash}")
print(f"Số chứa dấu cách: {count_space}")
print(f"Độ dài ngắn nhất (sau khi loại bỏ ký tự và NaN): {min_len}")
print(f"Độ dài dài nhất: {max_len}")


Số bắt đầu bằng '00': 0
Số bắt đầu bằng '+': 0
Số chứa dấu '-': 0
Số chứa dấu cách: 7000
Độ dài ngắn nhất (sau khi loại bỏ ký tự và NaN): 8
Độ dài dài nhất: 10


In [6]:
# Chuẩn hóa số điện thoại theo tiêu chuẩn E.164
from phonenumbers import PhoneNumberFormat
# ---- Country to Region Code Mapping ----
country_region_map = {
    "Sweden": "SE",
    "United Kingdom": "GB",
    "France": "FR",
    "Germany": "DE",
    "Poland": "PL",
    "Czechia": "CZ",
    "Slovakia": "SK",
    "Italy": "IT",
    "Spain": "ES",
    "Portugal": "PT",
    "Belgium": "BE",
    "Netherlands": "NL",
    "Hungary": "HU",
    "Austria": "AT"
}

# Hàm clean - giữ nan và xóa kí tự lạ (gồm dấu cách và - )
def clean_phone_number(raw_phone):
    if pd.isna(raw_phone):
        return raw_phone  
    raw_phone = str(raw_phone)
    cleaned = re.sub(r'[^\d+]', '', raw_phone)  
    return cleaned

# Chuẩn hóa số hợp lệ theo E.164 (thư viện phonenumbers để đưa về dạng sđt quốc tế)
def standardize_phone_number(row):
    raw = clean_phone_number(row["phone"])
    region = country_region_map.get(row["country"], None)
    if pd.isna(raw) or not str(raw).strip():
        return row["phone"]  
    try:
        parsed = phonenumbers.parse(raw, region)
        if phonenumbers.is_valid_number(parsed):
            return phonenumbers.format_number(parsed, PhoneNumberFormat.E164)
        else:
            return row["phone"]
    except:
        return row["phone"]

# Gán nhãn hợp lệ / không hợp lệ / thiếu để tiện lọc thủ công (nếu có)
def label_phone_status(row):
    raw = clean_phone_number(row["phone"])
    region = country_region_map.get(row["country"], None)
    if pd.isna(raw) or not str(raw).strip():
        return pd.NA 
    try:
        parsed = phonenumbers.parse(raw, region)
        if phonenumbers.is_valid_number(parsed):
            return 1  # Valid
        else:
            return 0  # Invalid
    except:
        return 0  # Invalid do lỗi

# Áp dụng hàm
df_checkpoint["phone"] = df_checkpoint.apply(standardize_phone_number, axis=1)
df_checkpoint["phone_status"] = df_checkpoint.apply(label_phone_status, axis=1)
print(df_checkpoint[["phone", "country", "phone_status"]].head(5))

          phone country phone_status
0  +33247643919  France            1
1  +33247728374  France            1
2  +33247202889  France            1
3           NaN  France         <NA>
4  +33247050528  France            1


In [8]:
df_checkpoint.head(5)

,post_title,address,latitude,longitude,phone,website,facebook,instagram,twitter,post_content,google_maps_link,google_profile,google_review_link,country,phone_status
0,Le Vietnam,"28 Rue Édouard Vaillant, 37000 Tours, France",47.389344,0.695498,+33247643919,NaN,NaN,NaN,NaN,"Le Vietnam located in 28 Rue Édouard Vaillant,...",https://maps.google.com/?cid=16278021701003570645,https://maps.google.com/?q=place_id:ChIJ2fvfyU...,https://search.google.com/local/reviews?placei...,France,1
1,Le Royaume d'Angkor,"17 Pl. des Halles, 37000 Tours, France",47.392128,0.679750,+33247728374,NaN,NaN,NaN,NaN,Le Royaume d'Angkor located in 17 Pl. des Hall...,https://maps.google.com/?cid=7510473987030531764,https://maps.google.com/?q=place_id:ChIJ2_FQS7...,https://search.google.com/local/reviews?placei...,France,1
2,La Chine rouge,"39 Rue Néricault Destouches, 37000 Tours, France",47.392259,0.684407,+33247202889,https://sites.google.com/view/lachinerouge,NaN,NaN,NaN,La cuisine chinoise\r\n中国菜 ( zhông guô cài ),https://maps.google.com/?cid=13044544289735649019,https://maps.google.com/?q=place_id:ChIJCUZs5L...,https://search.google.com/local/reviews?placei...,France,1
3,Pitaya Thaï Street Food,"8 Rue Raoul Follereau, 37100 Tours, France",47.425286,0.701363,NaN,https://pitaya-thaistreetfood.com/r/pitaya-tou...,NaN,NaN,NaN,PanierModifier-Votre panierAjouter un produitO...,https://maps.google.com/?cid=11273252203041383201,https://maps.google.com/?q=place_id:ChIJuyR_k8...,https://search.google.com/local/reviews?placei...,France,<NA>
4,L'INDOCHINE,"1 Place François Truffaut, 37000 Tours, France",47.388081,0.693375,+33247050528,http://www.restaurantindochine.fr/,NaN,NaN,NaN,"top of page1 Place François Truffaut, 37000 TO...",https://maps.google.com/?cid=12482834208217896565,https://maps.google.com/?q=place_id:ChIJ2bHnpr...,https://search.google.com/local/reviews?placei...,France,1


In [9]:
# tách address thành: street (có zip), zip (postcode), city
def robust_split_address(address, country="France"):
    if pd.isna(address):
        return "", "", ""

    # Bước 1: Xoá phần country ở cuối
    address = re.sub(
        rf'[,\s]*{re.escape(str(country))}[\s,\d]*$', '', address, flags=re.IGNORECASE
    ).strip()

    # Bước 2: Tìm postcode và city (giả định postcode là 123 45 hoặc 12345 hoặc 123-45)
    match = re.search(r'(\d{3}[-\s]?\d{2})\s+(.+)$', address)
    if match:
        zip_code = match.group(1).strip()
        city = match.group(2).strip()

        # Street là phần trước ZIP (có thể kèm dấu phẩy)
        street_part = address[:match.start()].strip().rstrip(', ')
        return street_part, zip_code, city

    # Nếu không tách được, coi toàn bộ là street
    return address, "", ""

# Áp dụng vào dữ liệu ban đầu
df_checkpoint[['street', 'zip', 'city']] = df_checkpoint['address'].apply(
    lambda x: pd.Series(robust_split_address(x))
)
# xóa cột address
df_checkpoint.drop(columns=['address'], inplace=True)


In [10]:
df_checkpoint.head(5)

,post_title,latitude,longitude,phone,website,facebook,instagram,twitter,post_content,google_maps_link,google_profile,google_review_link,country,phone_status,street,zip,city
0,Le Vietnam,47.389344,0.695498,+33247643919,NaN,NaN,NaN,NaN,"Le Vietnam located in 28 Rue Édouard Vaillant,...",https://maps.google.com/?cid=16278021701003570645,https://maps.google.com/?q=place_id:ChIJ2fvfyU...,https://search.google.com/local/reviews?placei...,France,1,28 Rue Édouard Vaillant,37000,Tours
1,Le Royaume d'Angkor,47.392128,0.679750,+33247728374,NaN,NaN,NaN,NaN,Le Royaume d'Angkor located in 17 Pl. des Hall...,https://maps.google.com/?cid=7510473987030531764,https://maps.google.com/?q=place_id:ChIJ2_FQS7...,https://search.google.com/local/reviews?placei...,France,1,17 Pl. des Halles,37000,Tours
2,La Chine rouge,47.392259,0.684407,+33247202889,https://sites.google.com/view/lachinerouge,NaN,NaN,NaN,La cuisine chinoise\r\n中国菜 ( zhông guô cài ),https://maps.google.com/?cid=13044544289735649019,https://maps.google.com/?q=place_id:ChIJCUZs5L...,https://search.google.com/local/reviews?placei...,France,1,39 Rue Néricault Destouches,37000,Tours
3,Pitaya Thaï Street Food,47.425286,0.701363,NaN,https://pitaya-thaistreetfood.com/r/pitaya-tou...,NaN,NaN,NaN,PanierModifier-Votre panierAjouter un produitO...,https://maps.google.com/?cid=11273252203041383201,https://maps.google.com/?q=place_id:ChIJuyR_k8...,https://search.google.com/local/reviews?placei...,France,<NA>,8 Rue Raoul Follereau,37100,Tours
4,L'INDOCHINE,47.388081,0.693375,+33247050528,http://www.restaurantindochine.fr/,NaN,NaN,NaN,"top of page1 Place François Truffaut, 37000 TO...",https://maps.google.com/?cid=12482834208217896565,https://maps.google.com/?q=place_id:ChIJ2bHnpr...,https://search.google.com/local/reviews?placei...,France,1,1 Place François Truffaut,37000,Tours


In [11]:
# Xuất file 
df_checkpoint.to_csv('fr_address_check_2.csv')

In [12]:
# Kiểm tra active của web 
import requests
from urllib.parse import urlparse
from concurrent.futures import ThreadPoolExecutor

# Chuẩn hóa URL
def normalize_url(url):
    if pd.isna(url) or not str(url).strip():
        return None
    url = url.strip()
    parsed = urlparse(url)
    if not parsed.scheme:
        return "http://" + url
    return url

# Kiểm tra hoạt động website, giữ original URL
def check_url(original_url):
    norm_url = normalize_url(original_url)
    if not norm_url:
        return (original_url, None, None, False)
    try:
        response = requests.get(norm_url, timeout=3, allow_redirects=True)
        final_url = response.url
        status = response.status_code
        is_active = 200 <= status < 400
        return (original_url, status, final_url, is_active)
    except:
        return (original_url, 0, None, False)

# Áp dụng đa luồng
df_checkpoint["normalized_url"] = df_checkpoint["website"].apply(normalize_url)
urls = df_checkpoint["normalized_url"].tolist()

with ThreadPoolExecutor(max_workers=30) as executor:
    results = list(executor.map(check_url, urls))

# Ghi kết quả vào DataFrame
df_checkpoint["web_status"] = [r[1] for r in results]
df_checkpoint["final_url"] = [r[2] for r in results]
df_checkpoint["is_active"] = [r[3] for r in results]


In [13]:
# Kiểm tra các link mạng xã hội bị lẫn trong website
# Xóa cột normalized_url
df_checkpoint.drop(columns=["normalized_url"], inplace=True)

# Xác định nền tảng mạng xã hội 
def classify_social_platform(url):
    if pd.isna(url):
        return None
    url = url.lower()
    if "facebook.com" in url:
        return "facebook"
    elif "instagram.com" in url:
        return "instagram"
    elif "twitter.com" in url or "x.com" in url:
        return "twitter"
    return None

df_checkpoint["social_platform"] = df_checkpoint["website"].apply(classify_social_platform)

# Chuyển các link sai về đúng cột
for platform in ["facebook", "instagram", "twitter"]:
    df_checkpoint[platform] = df_checkpoint.apply(
        lambda row: row["website"] if row["social_platform"] == platform and pd.isna(row[platform]) else row[platform],
        axis=1
    )

# Lưu kết quả vào file CSV
output_path = r"C:\Users\Nhung\Downloads\We_Love_Pho\Clean 25-5 - raw\fr_web_check.csv"

In [14]:
# Xoá khỏi giá trị cột website nếu là link MXH 
df_checkpoint.loc[df_checkpoint["social_platform"].notna(), "website"] = None
df_checkpoint.drop(columns=["social_platform"], inplace=True)

In [15]:
df_checkpoint.head(3)


,post_title,latitude,longitude,phone,website,facebook,instagram,twitter,post_content,google_maps_link,google_profile,google_review_link,country,phone_status,street,zip,city,web_status,final_url,is_active
0,Le Vietnam,47.389344,0.695498,+33247643919,NaN,NaN,NaN,NaN,"Le Vietnam located in 28 Rue Édouard Vaillant,...",https://maps.google.com/?cid=16278021701003570645,https://maps.google.com/?q=place_id:ChIJ2fvfyU...,https://search.google.com/local/reviews?placei...,France,1,28 Rue Édouard Vaillant,37000,Tours,NaN,None,False
1,Le Royaume d'Angkor,47.392128,0.679750,+33247728374,NaN,NaN,NaN,NaN,Le Royaume d'Angkor located in 17 Pl. des Hall...,https://maps.google.com/?cid=7510473987030531764,https://maps.google.com/?q=place_id:ChIJ2_FQS7...,https://search.google.com/local/reviews?placei...,France,1,17 Pl. des Halles,37000,Tours,NaN,None,False
2,La Chine rouge,47.392259,0.684407,+33247202889,https://sites.google.com/view/lachinerouge,NaN,NaN,NaN,La cuisine chinoise\r\n中国菜 ( zhông guô cài ),https://maps.google.com/?cid=13044544289735649019,https://maps.google.com/?q=place_id:ChIJCUZs5L...,https://search.google.com/local/reviews?placei...,France,1,39 Rue Néricault Destouches,37000,Tours,200.0,https://sites.google.com/view/lachinerouge,True


In [19]:
# tạo copy
df_checkpoint_copy = df_checkpoint.copy()

In [20]:
# 7. Lấy danh sách cột chuẩn từ file gd
gd_columns = df_gd.columns.tolist()

# 8. Thêm các cột còn thiếu và gán giá trị rỗng
for col in gd_columns:
    if col not in df_checkpoint_copy.columns:
        df_checkpoint_copy[col] = ""

# 9. Gán giá trị mặc định
df_checkpoint_copy['post_status'] = 'publish'
df_checkpoint_copy['post_category'] = ',249,'
df_checkpoint_copy['default_category'] = '249'
df_checkpoint_copy['featured'] = '0'

# 10. Sắp xếp lại đúng thứ tự cột
df_checkpoint_copy = df_checkpoint_copy[gd_columns]

# 12. Lưu file đã chuẩn hoá
df_checkpoint_copy.to_csv("fr_standardized_2.csv", index=False)

# 13. (Tuỳ chọn) Xem thử kết quả
print(df_checkpoint_copy[['street', 'zip', 'city', 'country']].head())

                        street    zip   city country
0      28 Rue Édouard Vaillant  37000  Tours  France
1            17 Pl. des Halles  37000  Tours  France
2  39 Rue Néricault Destouches  37000  Tours  France
3        8 Rue Raoul Follereau  37100  Tours  France
4    1 Place François Truffaut  37000  Tours  France


In [21]:
df_checkpoint_copy.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7245 entries, 0 to 7244
Data columns (total 30 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   ID                7245 non-null   object 
 1   post_title        7245 non-null   object 
 2   post_content      7245 non-null   object 
 3   post_status       7245 non-null   object 
 4   post_author       7245 non-null   object 
 5   post_type         7245 non-null   object 
 6   post_date         7245 non-null   object 
 7   post_modified     7245 non-null   object 
 8   post_tags         7245 non-null   object 
 9   post_category     7245 non-null   object 
 10  default_category  7245 non-null   object 
 11  featured          7245 non-null   object 
 12  street            7245 non-null   object 
 13  street2           7245 non-null   object 
 14  city              7245 non-null   object 
 15  region            7245 non-null   object 
 16  country           7245 non-null   object 


In [22]:
# in ra 5 giá trị đầu cột lat và long
print(df_checkpoint_copy[['latitude', 'longitude']].head())

    latitude  longitude
0  47.389344   0.695498
1  47.392128   0.679750
2  47.392259   0.684407
3  47.425286   0.701363
4  47.388081   0.693375


In [23]:
duplicates = df_checkpoint_copy[df_checkpoint_copy.duplicated(keep=False)]

In [24]:
duplicate_count = duplicates.shape[0]
print(f"Số lượng bản ghi trùng lặp: {duplicate_count}")    


Số lượng bản ghi trùng lặp: 7214


In [25]:
# Đếm số lượng giá trị không null cho từng dòng
df_checkpoint_copy['non_null_count'] = df_checkpoint_copy.notnull().sum(axis=1)

# Sắp xếp theo các cột và theo số lượng giá trị không null giảm dần
df_sorted = df_checkpoint_copy.sort_values(by=['post_title', 'latitude', 'longitude', 'street', 'non_null_count'], ascending=[True, True, True, True, False])

# Xóa các dòng trùng hoàn toàn, giữ lại dòng có nhiều thông tin nhất
df_deduplicated = df_sorted.drop_duplicates(keep='first').drop(columns=['non_null_count'])

# Lưu kết quả ra file mới
output_path = "fr_no_dup_2.csv"
df_deduplicated.to_csv(output_path, index=False)


In [26]:
df_deduplicated.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3638 entries, 2268 to 1137
Data columns (total 30 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   ID                3638 non-null   object 
 1   post_title        3638 non-null   object 
 2   post_content      3638 non-null   object 
 3   post_status       3638 non-null   object 
 4   post_author       3638 non-null   object 
 5   post_type         3638 non-null   object 
 6   post_date         3638 non-null   object 
 7   post_modified     3638 non-null   object 
 8   post_tags         3638 non-null   object 
 9   post_category     3638 non-null   object 
 10  default_category  3638 non-null   object 
 11  featured          3638 non-null   object 
 12  street            3638 non-null   object 
 13  street2           3638 non-null   object 
 14  city              3638 non-null   object 
 15  region            3638 non-null   object 
 16  country           3638 non-null   object 
 1

Top 200 từ khóa phổ biến và tần suất

In [28]:
%pip install nltk


Note: you may need to restart the kernel to use updated packages.


In [30]:
import nltk

In [35]:
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
# Kết hợp nội dung từ 2 cột post_title và post_content
text_data = df_deduplicated[['post_content']].fillna('').agg(' '.join, axis=1)

# Khởi tạo CountVectorizer để trích xuất từ khóa
french_stop_words = [
    'à', 'afin', 'alors', 'avant', 'avec', 'avoir', 'bon', 'car', 'ce', 'ceci', 'cela', 'ces',
    'dans', 'de', 'des', 'du', 'donc', 'elle', 'elles', 'en', 'encore', 'es', 'est', 'et',
    'étaient', 'étaient', 'être', 'eux', 'faire', 'fait', 'faites', 'fais', 'fois', 'font', 
    'il', 'ils', 'je', 'la', 'le', 'les', 'leur', 'lorsque', 'mais', 'me', 'même', 'mon', 
    'mot', 'nous', 'ou', 'par', 'parce', 'pas', 'pour', 'quand', 'que', 'quel', 'quelle', 
    'quelques', 'qui', 'sans', 'se', 'ses', 'si', 'son', 'sont', 'sur', 'te', 'tant', 'un',
    'une', 'votre', 'vous', 'vulnérable'
]
vectorizer = CountVectorizer(stop_words=french_stop_words, max_features=500)
X = vectorizer.fit_transform(text_data)

# Lấy ra từ và tần suất
keywords = vectorizer.get_feature_names_out()
frequencies = np.asarray(X.sum(axis=0)).flatten()

# Tạo dataframe kết quả
keywords_df = pd.DataFrame({'keyword': keywords, 'frequency': frequencies}).sort_values(by='frequency', ascending=False)

In [36]:
# Lưu kết quả vào file CSV
output_keywords_path = "fr_keywords_2.csv"
keywords_df.to_csv(output_keywords_path, index=False)

In [37]:
# Kiểm tra độ chính xác 
positive_keywords = [
    'vietnam', 'viet', 'việt', 'pho', 'bún', 'nem', 'saigon', "phở", 'sài gòn', 'hà nội', 'hanoi', 'bánh mì',
    'halong', 'huế', 'bánh', 'goi cuon', 'bun cha', 'banh', 'thang', 'nam', 'sen', 'hoan kiem', 'wietnam', 'vietnamese',
    'sapa', 'tre', 'ha long', 'ha noi', 'sai gon', 'sajgon', 'hoang', 'ha-noi', 'com tam', 'hoan', 'bami',
    'long', 'binh', 'banh mi', 'sao mai', 'song lam', 'ngoc', 'phuong dong', 'linh', 'vietnamská', 'vietnameské', 'quán', 'anh',
    'vietnamskou', 'vietnamské jídlo', 'vietnamská restaurace', 'ngon', 'hoi an', 'quan', 'vina', 'bếp', 'long', 'nón', 'hà', 
    'vietfood', 'gao', 'mì', 'mộc', 'thanh', 'cà', 'tuan', 'lá', 'rong', 'vietnamskou', 'vietnamu', 'vietnamesisches', 'chả',
    'vietnamesische küche', 'vietnamesisch', 'vietnamesischen', "baguette", "mì", "hội", "thêm", "moc", "hanoï", "mắm", "cô", "nguyen",
    "vietnamien", "vietnamienne", "saïgon", "bobun", "phuong", "bò", "vietfood", "cuốn", "phuoc", "saigonnais", "chào"
]
negative_keywords = [
    'chinese', 'thai', 'japan', 'korean', 'fusion', 'asia', 'china','india', 'ramen', 'pasta', 'pizza', 'burger',
    'sushi', 'tapas', 'mexican', 'indian', 'kebab', 'italian', 'curry', "tai wan", "singapore", "malaysia", "korea",
    "hong kong", 'resort', 'hotel', 'pub', 'cafe', 'coffee', 'steak', 'park', 'inn', 'post', 'market', 'hall', 'bbq',
    'library', 'sandwich', 'cantonese', 'peking', 'thajská', 'thajské', 'banyan', 'guty', 'shanghai', 'shi', 'pizzerie',
    'bubble', 'kyoto', "mongolian", "indochine", 'nail', 'spa', 'massage', 'laden', 'shop', 'store', 'beauty', 'thailändische',
    'asiatisch', 'asiatischen', 'café', "wagamama", "tuk", "coffee", "beauty", "giggling", "thailand", "beijing", "club", "bangkok",
    "halah", "pub", "theatre", "hungary", "mandarin", "yang", "castle", "casino", "kfc", 'thaï', "mandala", 'salon', "indien", 'thaïlandaises',
    "asiatique", "mékong", "hôtel", "angkor", "cantine", "mandarin", "bangkok", "thaïlandais", "phuket", "khmer", "parfum", "crêpes",
    "chinois", "wok", "japonais", "bas한국어", "chinoise", "pad", "spã", "cambodge", "patisserie", "bubble", "japonaises"
]

# Bước 3: Tạo regex pattern
pattern_positive = re.compile('|'.join(positive_keywords), re.IGNORECASE)
pattern_negative = re.compile('|'.join(negative_keywords), re.IGNORECASE)

# Bước 4: Hàm gán tag
def tag_positive(text):
    if pd.isna(text):
        return ''
    return 'Vietnamese restaurant' if pattern_positive.search(text) else ''

def tag_negative(text):
    if pd.isna(text):
        return ''
    return 'Others' if pattern_negative.search(text) else ''

# Bước 5: Gán PositiveTag và NegativeTag
df_deduplicated['Pos'] = df_deduplicated.apply(
    lambda row: tag_positive(row['post_title']) or tag_positive(row['post_content']),
    axis=1
)

df_deduplicated['Neg'] = df_deduplicated.apply(
    lambda row: tag_negative(row['post_title']) or tag_negative(row['post_content']),
    axis=1
)

# Bước 6: Logic gán ReCheck?
def final_recheck_tag(row):
    if row['Pos'] != '' and row['Neg'] == '':
        return 'N'
    elif row['Pos'] == '' and row['Neg'] != '':
        return 'N'
    elif row['Pos'] == '' and row['Neg'] == '':
        return 'Y'
    else:
        return 'Y'

df_deduplicated['ReCheck?'] = df_deduplicated.apply(final_recheck_tag, axis=1)

In [ ]:
# Tạo label mẫu
# Tạo 1 cột mới tên là rỗng mới là Y trong df_checkpoint
def assign_label(row):
    pos = row['Pos'] == 'Vietnamese restaurant'
    neg = row['Neg'] == 'Others'
    recheck = row['ReCheck?']

    if pos and not neg and recheck == 'N':
        return 1
    elif neg and not pos and recheck == 'N':
        return 0
    elif pos and neg and recheck == 'Y':
        return 0
    else:
        return ''

df_deduplicated['Y'] = df_deduplicated.apply(assign_label, axis=1)

df = df_deduplicated.copy()

# Xuất file để check manual
columns_to_export = [
    'post_title', 'post_content', 'website', 'google_profile', "Y", 'city'
]
df = df[columns_to_export]
df.to_csv('fr_labeled_2.csv', index=False)


In [39]:
df_deduplicated.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3638 entries, 2268 to 1137
Data columns (total 34 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   ID                3638 non-null   object 
 1   post_title        3638 non-null   object 
 2   post_content      3638 non-null   object 
 3   post_status       3638 non-null   object 
 4   post_author       3638 non-null   object 
 5   post_type         3638 non-null   object 
 6   post_date         3638 non-null   object 
 7   post_modified     3638 non-null   object 
 8   post_tags         3638 non-null   object 
 9   post_category     3638 non-null   object 
 10  default_category  3638 non-null   object 
 11  featured          3638 non-null   object 
 12  street            3638 non-null   object 
 13  street2           3638 non-null   object 
 14  city              3638 non-null   object 
 15  region            3638 non-null   object 
 16  country           3638 non-null   object 
 1

In [40]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3638 entries, 2268 to 1137
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   post_title      3638 non-null   object
 1   post_content    3638 non-null   object
 2   website         1749 non-null   object
 3   google_profile  3638 non-null   object
 4   Y               3638 non-null   object
 5   city            3638 non-null   object
dtypes: object(6)
memory usage: 199.0+ KB


In [41]:
# Đọc file labeled
df_labeled = pd.read_csv('fr_labeled.csv')
# Kiểm tra số nhãn của từng unique giá trị trong cột 'Y'
print(df_labeled['Y'].value_counts(dropna=False))

Y
0.0    1470
NaN    1105
1.0    1063
Name: count, dtype: int64


In [42]:
print(df_deduplicated['Y'].value_counts(dropna=False))

Y
0    1470
     1105
1    1063
Name: count, dtype: int64


In [43]:
# Kiểm tra lại nhanh số lượng giá trị thiếu (NaN) sau khi chuyển đổi
missing_summary = df_deduplicated.isna().sum()
missing_summary

ID                     0
post_title             0
post_content           0
post_status            0
post_author            0
post_type              0
post_date              0
post_modified          0
post_tags              0
post_category          0
default_category       0
featured               0
street                 0
street2                0
city                   0
region                 0
country                0
zip                    0
latitude               0
longitude              0
website             1889
neighbourhood          0
facebook            3294
instagram           3576
twitter             3630
phone                123
email                  0
logo                   0
google_profile         0
post_images            0
Pos                    0
Neg                    0
ReCheck?               0
Y                      0
dtype: int64

In [44]:
df_final = df_deduplicated.copy()

In [45]:
df_final[['phone']].head(10)


,phone
2268,+33470282765
2213,+33758572166
357,+33698092143
3126,+33780993838
3121,+33780993838
3195,+33780993838
1903,+33745405058
2510,+33380430470
2271,+33470299918
3592,+33988465300


In [57]:
df_final['phone'] = df_final['phone'].apply(lambda x: str(int(x)) if pd.notna(x) else "")
df_final[['phone']].head(10)

ValueError: invalid literal for int() with base 10: '28 99 98 41'

In [46]:
df_final.head(5)

,ID,post_title,post_content,post_status,post_author,post_type,post_date,post_modified,post_tags,post_category,...,twitter,phone,email,logo,google_profile,post_images,Pos,Neg,ReCheck?,Y
2268,,03 Wok Panda,Bienvenue chez le site officiel de Wok Panda d...,publish,,,,,,",249,",...,NaN,+33470282765,,,https://maps.google.com/?q=place_id:ChIJIxZ0bw...,,,Others,N,0
2213,,10 Onnumara Çiğköfte,10 Onnumara Çiğköfte located in 23 Rue Anatole...,publish,,,,,,",249,",...,NaN,+33758572166,,,https://maps.google.com/?q=place_id:ChIJHQ2ZyI...,,,,Y,
357,,2 l'Asie à l'Orient & Food truck,"Un voyage culinaire de l'Asie a l'Orient , de ...",publish,,,,,,",249,",...,NaN,+33698092143,,,https://maps.google.com/?q=place_id:ChIJ91u-KY...,,Vietnamese restaurant,,N,1
3126,,38° Tea - Bubble Tea & Waffle ANTONY,Venez déguster nos délicieux Bubble Tea dans l...,publish,,,,,,",249,",...,NaN,+33780993838,,,https://maps.google.com/?q=place_id:ChIJo9uJkP...,,,Others,N,0
3121,,38° Tea - Bubble Tea & Waffle PALAISEAU,Venez déguster nos délicieux Bubble Tea dans l...,publish,,,,,,",249,",...,NaN,+33780993838,,,https://maps.google.com/?q=place_id:ChIJ72oBcf...,,,Others,N,0


In [47]:
# Gán giá trị mặc định nếu thiếu
df_final['post_type'] = df_final['post_type'].fillna('gd_place')
# Gán ngày mặc định nếu thiếu, đảm bảo đúng định dạng chuỗi
default_date = '2025-06-06 00:00:00'
df_final['post_date'] = df_final['post_date'].fillna(default_date)
df_final['post_modified'] = df_final['post_modified'].fillna(default_date)
df_final['post_author'] = df_final['post_author'].fillna('admin')
df_final["post_content"] = ""
df_final = df_final.replace(r'^\s*$', pd.NA, regex=True)
df_final.set_index('ID', inplace=True)
# Đổi tên cột id thành ID
df_final.head(5)
df_final.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3638 entries, <NA> to <NA>
Data columns (total 33 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   post_title        3638 non-null   object 
 1   post_content      0 non-null      object 
 2   post_status       3638 non-null   object 
 3   post_author       0 non-null      object 
 4   post_type         0 non-null      object 
 5   post_date         0 non-null      object 
 6   post_modified     0 non-null      object 
 7   post_tags         0 non-null      object 
 8   post_category     3638 non-null   object 
 9   default_category  3638 non-null   object 
 10  featured          3638 non-null   object 
 11  street            3626 non-null   object 
 12  street2           0 non-null      object 
 13  city              3632 non-null   object 
 14  region            0 non-null      object 
 15  country           3638 non-null   object 
 16  zip               3632 non-null   object 
 1

In [48]:
df_final[['phone']].head(10)

,phone
ID,
<NA>,+33470282765
<NA>,+33758572166
<NA>,+33698092143
<NA>,+33780993838
<NA>,+33780993838
<NA>,+33780993838
<NA>,+33745405058
<NA>,+33380430470
<NA>,+33470299918


In [49]:
df_final.head(5)
df_final.to_csv('fr_final_final_.csv', index=False)

In [ ]:
%pip install openpyxl --quiet

# Xuất file excel
output_excel_path = r"fr_final_final_2.csv"
df_final.to_excel(output_excel_path, index=False)

Note: you may need to restart the kernel to use updated packages.


ValueError: No engine for filetype: 'csv'

In [50]:
# đọc file labeled_new
#df_labeled_new = pd.read_csv('ger_labeled_new.csv')
df_standardized= pd.read_csv('fr_final_final_.csv')
# Kiểm tra số nhãn của từng unique giá trị trong cột 'Y'
print(df_standardized['Y'].value_counts(dropna=False))

Y
0.0    1374
NaN    1283
1.0     981
Name: count, dtype: int64


In [51]:
df_standardized['phone'] = df_standardized['phone'].apply(lambda x: str(int(x)) if pd.notna(x) else "")
df_standardized[['phone']].head(10)

ValueError: invalid literal for int() with base 10: '05 90 84 09 53'

In [52]:
df_standardized.head(5)

,post_title,post_content,post_status,post_author,post_type,post_date,post_modified,post_tags,post_category,default_category,...,twitter,phone,email,logo,google_profile,post_images,Pos,Neg,ReCheck?,Y
0,03 Wok Panda,NaN,publish,NaN,NaN,NaN,NaN,NaN,",249,",249,...,NaN,33470282765,NaN,NaN,https://maps.google.com/?q=place_id:ChIJIxZ0bw...,NaN,NaN,Others,N,0.0
1,10 Onnumara Çiğköfte,NaN,publish,NaN,NaN,NaN,NaN,NaN,",249,",249,...,NaN,33758572166,NaN,NaN,https://maps.google.com/?q=place_id:ChIJHQ2ZyI...,NaN,NaN,NaN,Y,NaN
2,2 l'Asie à l'Orient & Food truck,NaN,publish,NaN,NaN,NaN,NaN,NaN,",249,",249,...,NaN,33698092143,NaN,NaN,https://maps.google.com/?q=place_id:ChIJ91u-KY...,NaN,Vietnamese restaurant,NaN,N,NaN
3,38° Tea - Bubble Tea & Waffle ANTONY,NaN,publish,NaN,NaN,NaN,NaN,NaN,",249,",249,...,NaN,33780993838,NaN,NaN,https://maps.google.com/?q=place_id:ChIJo9uJkP...,NaN,NaN,Others,N,0.0
4,38° Tea - Bubble Tea & Waffle PALAISEAU,NaN,publish,NaN,NaN,NaN,NaN,NaN,",249,",249,...,NaN,33780993838,NaN,NaN,https://maps.google.com/?q=place_id:ChIJ72oBcf...,NaN,NaN,Others,N,0.0


In [53]:
# Bổ sung giá trị cột Y từ df_labeled_new vào df_standardized dựa trên index (không có cột ID)
df_labeled_new = pd.read_csv(r"C:\Users\Nhung\Downloads\We_Love_Pho\fr_final_final_.csv")
df_standardized['Y'] = df_labeled_new['Y'].values
# Lưu kết quả vào file CSV
output_path = 'fr_standardized_with_labels_2.csv'
# đọc poland_standardized_with_labels.csv
df_standardized.to_csv(output_path, index=False)
df_standardized_with_labels = pd.read_csv(output_path)  
# Kiểm tra số nhãn của từng unique giá trị trong cột 'Y'
print(df_standardized_with_labels['Y'].value_counts(dropna=False))
# Lưu lại file đã chuẩn hoá
df_standardized_with_labels.to_csv('fr_standardized_final_2.csv', index=False)
# in head 5 dòng
df_standardized_with_labels.head(5)

Y
0.0    1374
NaN    1283
1.0     981
Name: count, dtype: int64


,post_title,post_content,post_status,post_author,post_type,post_date,post_modified,post_tags,post_category,default_category,...,twitter,phone,email,logo,google_profile,post_images,Pos,Neg,ReCheck?,Y
0,03 Wok Panda,NaN,publish,NaN,NaN,NaN,NaN,NaN,",249,",249,...,NaN,33470282765,NaN,NaN,https://maps.google.com/?q=place_id:ChIJIxZ0bw...,NaN,NaN,Others,N,0.0
1,10 Onnumara Çiğköfte,NaN,publish,NaN,NaN,NaN,NaN,NaN,",249,",249,...,NaN,33758572166,NaN,NaN,https://maps.google.com/?q=place_id:ChIJHQ2ZyI...,NaN,NaN,NaN,Y,NaN
2,2 l'Asie à l'Orient & Food truck,NaN,publish,NaN,NaN,NaN,NaN,NaN,",249,",249,...,NaN,33698092143,NaN,NaN,https://maps.google.com/?q=place_id:ChIJ91u-KY...,NaN,Vietnamese restaurant,NaN,N,NaN
3,38° Tea - Bubble Tea & Waffle ANTONY,NaN,publish,NaN,NaN,NaN,NaN,NaN,",249,",249,...,NaN,33780993838,NaN,NaN,https://maps.google.com/?q=place_id:ChIJo9uJkP...,NaN,NaN,Others,N,0.0
4,38° Tea - Bubble Tea & Waffle PALAISEAU,NaN,publish,NaN,NaN,NaN,NaN,NaN,",249,",249,...,NaN,33780993838,NaN,NaN,https://maps.google.com/?q=place_id:ChIJ72oBcf...,NaN,NaN,Others,N,0.0


In [54]:
df_standardized = pd.read_csv(r"C:\Users\Nhung\Downloads\We_Love_Pho\fr_standardized_final_2.csv")


In [55]:
# Kiểm tra số nhãn của từng unique giá trị trong cột 'Y'
print(df_standardized['Y'].value_counts(dropna=False))

Y
0.0    1374
NaN    1283
1.0     981
Name: count, dtype: int64


In [56]:
# Gán giá trị mặc định nếu thiếu
df_standardized['post_type'] = df_standardized['post_type'].fillna('gd_place')
# Gán ngày mặc định nếu thiếu, đảm bảo đúng định dạng chuỗi
default_date = '2025-06-06 00:00:00'
df_standardized['post_date'] = df_standardized['post_date'].fillna(default_date)
df_standardized['post_modified'] = df_standardized['post_modified'].fillna(default_date)
df_standardized['post_author'] = df_standardized['post_author'].fillna('admin')
df_standardized["post_content"] = ""
df_standardized = df_standardized.replace(r'^\s*$', pd.NA, regex=True)
# trích xuất file cuối chỉ có các dòng mà giá trị cột Y là 1 và xóa cột Y sau đó 
df_standardized = df_standardized[df_standardized['Y'] == 1]
df_standardized.drop(columns=['Y'], inplace=True)
df_standardized.head(5)

,post_title,post_content,post_status,post_author,post_type,post_date,post_modified,post_tags,post_category,default_category,...,instagram,twitter,phone,email,logo,google_profile,post_images,Pos,Neg,ReCheck?
13,9 PHO,<NA>,publish,admin,gd_place,2025-06-06 00:00:00,2025-06-06 00:00:00,NaN,",249,",249,...,NaN,NaN,33146374219,NaN,NaN,https://maps.google.com/?q=place_id:ChIJe68nqy...,NaN,Vietnamese restaurant,NaN,N
26,AOZAI,<NA>,publish,admin,gd_place,2025-06-06 00:00:00,2025-06-06 00:00:00,NaN,",249,",249,...,NaN,NaN,33534564030,NaN,NaN,https://maps.google.com/?q=place_id:ChIJ1cIB5R...,NaN,Vietnamese restaurant,NaN,N
39,AU P'TIT HANOI,<NA>,publish,admin,gd_place,2025-06-06 00:00:00,2025-06-06 00:00:00,NaN,",249,",249,...,NaN,NaN,33556520729,NaN,NaN,https://maps.google.com/?q=place_id:ChIJ4cTSoQ...,NaN,Vietnamese restaurant,NaN,N
42,AU VIEUX HANOI,<NA>,publish,admin,gd_place,2025-06-06 00:00:00,2025-06-06 00:00:00,NaN,",249,",249,...,NaN,NaN,33143275080,NaN,NaN,https://maps.google.com/?q=place_id:ChIJ0cnBCg...,NaN,Vietnamese restaurant,NaN,N
49,Ai Pho,<NA>,publish,admin,gd_place,2025-06-06 00:00:00,2025-06-06 00:00:00,NaN,",249,",249,...,NaN,NaN,33171401450,NaN,NaN,https://maps.google.com/?q=place_id:ChIJLfecML...,NaN,Vietnamese restaurant,NaN,N


In [57]:
# So sánh định dạng từng cột của df_true với df_gd
def compare_column_formats(df1, df2):
    comparison = {}
    for col in df1.columns:
        if col in df2.columns:
            comparison[col] = {
                'df1_dtype': df1[col].dtype,
                'df2_dtype': df2[col].dtype,
                'df1_unique_count': df1[col].nunique(),
                'df2_unique_count': df2[col].nunique()
            }
        else:
            comparison[col] = {
                'df1_dtype': df1[col].dtype,
                'df2_dtype': None,
                'df1_unique_count': df1[col].nunique(),
                'df2_unique_count': None
            }
    return comparison
# So sánh định dạng cột của df_true với df_gd
comparison_result = compare_column_formats(df_standardized, df_gd)
# In kết quả so sánh
for col, info in comparison_result.items():
    print(f"Cột: {col}")
    print(f"  - df_true dtype: {info['df1_dtype']}, unique count: {info['df1_unique_count']}")
    print(f"  - df_gd dtype: {info['df2_dtype']}, unique count: {info['df2_unique_count']}")
    print()
# Ép kiểu các cột trong df_true để phù hợp với df_gd
def convert_column_types(df, reference_df):
    for col in reference_df.columns:
        if col in df.columns:
            ref_dtype = reference_df[col].dtype
            if ref_dtype == 'object':
                df[col] = df[col].astype(str)
            elif ref_dtype == 'int64':
                df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)
            elif ref_dtype == 'float64':
                df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0.0).astype(float)
            elif ref_dtype == 'datetime64[ns]':
                df[col] = pd.to_datetime(df[col], errors='coerce')
    return df   
# Chuyển đổi kiểu dữ liệu của df_true để phù hợp với df_gd
df_standardized = convert_column_types(df_standardized, df_gd)

Cột: post_title
  - df_true dtype: object, unique count: 837
  - df_gd dtype: object, unique count: 100

Cột: post_content
  - df_true dtype: object, unique count: 0
  - df_gd dtype: object, unique count: 19

Cột: post_status
  - df_true dtype: object, unique count: 1
  - df_gd dtype: object, unique count: 1

Cột: post_author
  - df_true dtype: object, unique count: 1
  - df_gd dtype: int64, unique count: 19

Cột: post_type
  - df_true dtype: object, unique count: 1
  - df_gd dtype: object, unique count: 1

Cột: post_date
  - df_true dtype: object, unique count: 1
  - df_gd dtype: object, unique count: 100

Cột: post_modified
  - df_true dtype: object, unique count: 1
  - df_gd dtype: object, unique count: 100

Cột: post_tags
  - df_true dtype: float64, unique count: 0
  - df_gd dtype: object, unique count: 1

Cột: post_category
  - df_true dtype: object, unique count: 1
  - df_gd dtype: object, unique count: 1

Cột: default_category
  - df_true dtype: int64, unique count: 1
  - df_gd 

In [144]:
# trích xuất file cuối chỉ có các dòng mà giá trị cột Y là 1 và xóa cột Y sau đó 
df_standardized = df_standardized[df_standardized['Y'] == 1]
df_standardized.drop(columns=['Y'], inplace=True)
df_standardized.head(5)

,ID,post_title,post_content,post_status,post_author,post_type,post_date,post_modified,post_tags,post_category,...,instagram,twitter,phone,email,logo,google_profile,post_images,Pos,Neg,ReCheck?
6,NaN,3 Mien,"3 Mien located in 64 Middlesex St, London E1 7...",publish,NaN,NaN,NaN,NaN,NaN,",249,",...,NaN,NaN,442072473344,NaN,NaN,https://maps.google.com/?q=place_id:ChIJud1qFx...,NaN,NaN,NaN,Y
8,NaN,4 Seasons Tree,"4 Seasons Tree located in 52 North Rd, Durham ...",publish,NaN,NaN,NaN,NaN,NaN,",249,",...,NaN,NaN,,NaN,NaN,https://maps.google.com/?q=place_id:ChIJwxGx_I...,NaN,Vietnamese restaurant,NaN,N
29,NaN,Amie's Kitchen,Amies Kitchen Vietnamese restaurant dine in an...,publish,NaN,NaN,NaN,NaN,NaN,",249,",...,NaN,NaN,447877598003,NaN,NaN,https://maps.google.com/?q=place_id:ChIJK_BLGj...,NaN,Vietnamese restaurant,NaN,N
30,NaN,Amthuc Viet,Home of Amthuc Viet. A Vietnamese catering & t...,publish,NaN,NaN,NaN,NaN,NaN,",249,",...,NaN,NaN,447900307025,NaN,NaN,https://maps.google.com/?q=place_id:ChIJNxj_s3...,NaN,Vietnamese restaurant,NaN,N
31,NaN,An Nam Restaurant,HomeAbout UsMenuGalleryReservationContact UsHo...,publish,NaN,NaN,NaN,NaN,NaN,",249,",...,NaN,NaN,442081434225,NaN,NaN,https://maps.google.com/?q=place_id:ChIJ8yXry4...,NaN,Vietnamese restaurant,Others,Y


In [58]:
df_standardized[['phone']].head(10)

,phone
13,33146374219
26,33534564030
39,33556520729
42,33143275080
49,33171401450
54,33978801291
60,33427411468
61,33673424850
62,33561837032
63,33556177324


In [59]:
def check_duplicates_multiple_columns(df1, df2, columns):
    merged = df1.merge(df2[columns].drop_duplicates(), on=columns, how='inner')
    return merged

# Gọi với 2 cột
duplicates = check_duplicates_multiple_columns(df_standardized, df_gd, [ 'street'])

print(f"Số lượng bản ghi trùng lặp theo 'post_title' và 'country': {duplicates.shape[0]}")
print(duplicates[['post_title', 'zip']].head(5))


Số lượng bản ghi trùng lặp theo 'post_title' và 'country': 3
              post_title      zip
0  Dong Huong Restaurant  75011.0
1            Ha Noi 1988  75001.0
2                Phở Tài  75013.0


In [60]:
# Xóa các bản ghi trùng lặp trong df_standardized_with_labels
df_standardized = df_standardized[~df_standardized['post_title'].isin(duplicates['post_title'])]
# In ra số lượng bản ghi sau khi xóa trùng lặp
print(f"Số lượng bản ghi sau khi xóa trùng lặp: {df_standardized.shape[0]}")

Số lượng bản ghi sau khi xóa trùng lặp: 978


In [61]:
df_standardized.head(5)
#Xóa cột neg, pos, recheck
df_standardized.drop(columns=['Neg', 'Pos', 'ReCheck?'], inplace=True)
# In ra thông tin của các cột
df_standardized.info()
# Lưu lại file đã chuẩn hoá
df_standardized.to_csv('fr_upload_2.csv', index=False)

<class 'pandas.core.frame.DataFrame'>
Index: 978 entries, 13 to 3633
Data columns (total 29 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   post_title        978 non-null    object 
 1   post_content      978 non-null    object 
 2   post_status       978 non-null    object 
 3   post_author       978 non-null    int64  
 4   post_type         978 non-null    object 
 5   post_date         978 non-null    object 
 6   post_modified     978 non-null    object 
 7   post_tags         978 non-null    object 
 8   post_category     978 non-null    object 
 9   default_category  978 non-null    int64  
 10  featured          978 non-null    int64  
 11  street            978 non-null    object 
 12  street2           978 non-null    float64
 13  city              978 non-null    object 
 14  region            978 non-null    object 
 15  country           978 non-null    object 
 16  zip               978 non-null    object 
 17  

In [63]:
df_standardized.head(5)

,post_title,post_content,post_status,post_author,post_type,post_date,post_modified,post_tags,post_category,default_category,...,website,neighbourhood,facebook,instagram,twitter,phone,email,logo,google_profile,post_images
13,9 PHO,<NA>,publish,0,gd_place,2025-06-06 00:00:00,2025-06-06 00:00:00,<NA>,",249,",249,...,https://9pho.fr/,0.0,<NA>,<NA>,<NA>,+33146374219,<NA>,<NA>,https://maps.google.com/?q=place_id:ChIJe68nqy...,0.0
26,AOZAI,<NA>,publish,0,gd_place,2025-06-06 00:00:00,2025-06-06 00:00:00,<NA>,",249,",249,...,<NA>,0.0,https://facebook.com/restaurantaozai,<NA>,<NA>,+33534564030,<NA>,<NA>,https://maps.google.com/?q=place_id:ChIJ1cIB5R...,0.0
39,AU P'TIT HANOI,<NA>,publish,0,gd_place,2025-06-06 00:00:00,2025-06-06 00:00:00,<NA>,",249,",249,...,https://auptithanoi.fr/,0.0,<NA>,<NA>,<NA>,+33556520729,<NA>,<NA>,https://maps.google.com/?q=place_id:ChIJ4cTSoQ...,0.0
42,AU VIEUX HANOI,<NA>,publish,0,gd_place,2025-06-06 00:00:00,2025-06-06 00:00:00,<NA>,",249,",249,...,https://auvieuxhanoi.com/,0.0,<NA>,<NA>,<NA>,+33143275080,<NA>,<NA>,https://maps.google.com/?q=place_id:ChIJ0cnBCg...,0.0
49,Ai Pho,<NA>,publish,0,gd_place,2025-06-06 00:00:00,2025-06-06 00:00:00,<NA>,",249,",249,...,https://aipho.fr/,0.0,<NA>,<NA>,<NA>,+33171401450,<NA>,<NA>,https://maps.google.com/?q=place_id:ChIJLfecML...,0.0


In [64]:
import pandas as pd

# 1. Load dữ liệu
df_slovakia = pd.read_csv(r"C:\Users\Nhung\Downloads\We_Love_Pho\fr_upload_2.csv")
df_sample = pd.read_csv(r"C:\Users\Nhung\Downloads\We_Love_Pho\sample structure.csv")

# 2. Thêm cột ID từ index (bắt đầu từ 1)
df_slovakia['ID'] = df_slovakia.index + 1
df_slovakia = df_slovakia[['ID'] + [col for col in df_slovakia.columns if col != 'ID']]

# 3. Đảm bảo các cột đúng thứ tự như file mẫu
df_slovakia = df_slovakia[df_sample.columns]

df_slovakia["post_type"] = "gd_place"
# Gán ngày mặc định nếu thiếu, đảm bảo đúng định dạng chuỗi
df_slovakia['post_date'] = '2025-06-06 00:00:00'
df_slovakia['post_modified'] = '2025-06-06 00:00:00'
df_slovakia['post_author'] = 'admin'
df_slovakia["post_content"] = ''

# 4. Chuyển kiểu dữ liệu theo sample
for col in df_sample.columns:
    ref_dtype = df_sample[col].dtype
    if ref_dtype == 'object':
        df_slovakia[col] = df_slovakia[col].astype(str)
    elif 'int' in str(ref_dtype):
        df_slovakia[col] = pd.to_numeric(df_slovakia[col], errors='coerce').fillna(0).astype(int)
    elif 'float' in str(ref_dtype):
        df_slovakia[col] = pd.to_numeric(df_slovakia[col], errors='coerce')
    elif 'datetime' in str(ref_dtype):
        df_slovakia[col] = pd.to_datetime(df_slovakia[col], errors='coerce')

# 5. Làm sạch các chuỗi rỗng hoặc chứa 'nan', 'none'
df_slovakia = df_slovakia.replace(r'^\s*$', pd.NA, regex=True)
df_slovakia = df_slovakia.applymap(lambda x: pd.NA if isinstance(x, str) and x.strip().lower() in ['nan', 'none'] else x)

df_slovakia['region'] = df_slovakia['region'].apply(
    lambda x: "-" if pd.isna(x) or str(x).strip().lower() in ['0.0', 'nan', 'none', 'n/a'] else x
)
# Chuẩn hóa zip code
df_slovakia['zip'] = df_slovakia['zip'].astype(str).str.replace(r'\.0$', '', regex=True)
df_slovakia['zip'] = df_slovakia['zip'].apply(lambda x: x.zfill(5) if x.isdigit() else x)
# Làm sạch street2 để không có 0.0 hoặc NaN
df_slovakia['street2'] = df_slovakia['street2'].apply(
    lambda x: pd.NA if pd.isna(x) or str(x).strip().lower() in ['0.0', 'nan', 'none'] else x
)

# 6. Chuẩn hóa số điện thoại
df_slovakia['phone'] = df_slovakia['phone'].astype(str).str.replace(r'\.0$', '', regex=True)
df_slovakia['phone'] = df_slovakia['phone'].apply(lambda x: '+' + x if isinstance(x, str) and x and not x.startswith('+') else x)

df_slovakia.drop(columns=['ID'], inplace=True)
# 8. Xuất ra file CSV
df_slovakia.to_csv("fr_final_upload_ready_2.csv", index=False)


C:\Users\Nhung\AppData\Local\Temp\ipykernel_42388\1596226467.py:35: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_slovakia = df_slovakia.applymap(lambda x: pd.NA if isinstance(x, str) and x.strip().lower() in ['nan', 'none'] else x)


In [65]:
# 1. Lọc các bản ghi trùng lặp theo post_title, country và google_profile
duplicates = df_slovakia[df_slovakia.duplicated(subset=['post_title', 'country', 'google_profile'], keep=False)]

# 2. In ra post_title của các bản ghi trùng lặp
print(duplicates['post_title'].unique())


['Chez Cô Hai' 'Saigon - Traiteur Vietnamien']


In [68]:
# 1. Xóa các bản ghi trùng lặp theo post_title, country và google_profile, chỉ giữ lại bản ghi đầu tiên
df_slovakia = df_slovakia.drop_duplicates(subset=['post_title', 'country', 'google_profile'], keep='first')

# 2. Kiểm tra và in kết quả
print(f"Số lượng bản ghi sau khi xóa trùng lặp: {len(df_slovakia)}")

# 3. Xuất kết quả ra file CSV sau khi xóa trùng lặp
df_slovakia.to_csv("fr_final_upload_ready_2.csv", index=False)


Số lượng bản ghi sau khi xóa trùng lặp: 976


In [72]:
# Let's first load the two uploaded files and perform the necessary steps to remove duplicates from df_slovakia
import pandas as pd

# Load the data from the uploaded files
df_slovakia = pd.read_csv(r"C:\Users\Nhung\Downloads\We_Love_Pho\fr_final_upload_ready_2.csv")
df_sample = pd.read_csv(r"C:\Users\Nhung\Downloads\We_Love_Pho\fr\fr_final_upload_ready.csv")


# Normalize the data types for comparison
for col in df_slovakia.columns:
    if df_slovakia[col].dtype != df_sample[col].dtype:
        df_slovakia[col] = df_slovakia[col].astype(str)
        df_sample[col] = df_sample[col].astype(str)

# Perform a merge to identify the common rows (rows that are exactly the same)
common_rows = pd.merge(df_slovakia, df_sample, on=df_slovakia.columns.tolist(), how='inner')

# Remove the common rows from df_slovakia
df_slovakia_cleaned = df_slovakia[~df_slovakia.apply(tuple, 1).isin(common_rows.apply(tuple, 1))]

# Check how many rows were removed (which are the duplicates)
removed_rows_count = len(common_rows)

removed_rows_count, df_slovakia_cleaned.shape[0]


(27, 949)

In [73]:
import pandas as pd

# 1. Load dữ liệu
df_slovakia = pd.read_csv(r"C:\Users\Nhung\Downloads\We_Love_Pho\fr_final_upload_ready_2.csv")
df_sample = pd.read_csv(r"C:\Users\Nhung\Downloads\We_Love_Pho\sample structure.csv")

# 2. Thêm cột ID từ index (bắt đầu từ 1)
df_slovakia['ID'] = df_slovakia.index + 1
df_slovakia = df_slovakia[['ID'] + [col for col in df_slovakia.columns if col != 'ID']]

# 3. Đảm bảo các cột đúng thứ tự như file mẫu
df_slovakia = df_slovakia[df_sample.columns]

df_slovakia["post_type"] = "gd_place"
# Gán ngày mặc định nếu thiếu, đảm bảo đúng định dạng chuỗi
df_slovakia['post_date'] = '2025-06-06 00:00:00'
df_slovakia['post_modified'] = '2025-06-06 00:00:00'
df_slovakia['post_author'] = 'admin'
df_slovakia["post_content"] = ''

# 4. Chuyển kiểu dữ liệu theo sample
for col in df_sample.columns:
    ref_dtype = df_sample[col].dtype
    if ref_dtype == 'object':
        df_slovakia[col] = df_slovakia[col].astype(str)
    elif 'int' in str(ref_dtype):
        df_slovakia[col] = pd.to_numeric(df_slovakia[col], errors='coerce').fillna(0).astype(int)
    elif 'float' in str(ref_dtype):
        df_slovakia[col] = pd.to_numeric(df_slovakia[col], errors='coerce')
    elif 'datetime' in str(ref_dtype):
        df_slovakia[col] = pd.to_datetime(df_slovakia[col], errors='coerce')

# 5. Làm sạch các chuỗi rỗng hoặc chứa 'nan', 'none'
df_slovakia = df_slovakia.replace(r'^\s*$', pd.NA, regex=True)
df_slovakia = df_slovakia.applymap(lambda x: pd.NA if isinstance(x, str) and x.strip().lower() in ['nan', 'none'] else x)

df_slovakia['region'] = df_slovakia['region'].apply(
    lambda x: "-" if pd.isna(x) or str(x).strip().lower() in ['0.0', 'nan', 'none', 'n/a'] else x
)
# Chuẩn hóa zip code
df_slovakia['zip'] = df_slovakia['zip'].astype(str).str.replace(r'\.0$', '', regex=True)
df_slovakia['zip'] = df_slovakia['zip'].apply(lambda x: x.zfill(5) if x.isdigit() else x)
# Làm sạch street2 để không có 0.0 hoặc NaN
df_slovakia['street2'] = df_slovakia['street2'].apply(
    lambda x: pd.NA if pd.isna(x) or str(x).strip().lower() in ['0.0', 'nan', 'none'] else x
)

# 6. Chuẩn hóa số điện thoại
df_slovakia['phone'] = df_slovakia['phone'].astype(str).str.replace(r'\.0$', '', regex=True)
df_slovakia['phone'] = df_slovakia['phone'].apply(lambda x: '+' + x if isinstance(x, str) and x and not x.startswith('+') else x)

df_slovakia.drop(columns=['ID'], inplace=True)
# 8. Xuất ra file CSV
df_slovakia.to_csv("fr_final_upload_ready_2.csv", index=False)


C:\Users\Nhung\AppData\Local\Temp\ipykernel_42388\3373423649.py:35: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_slovakia = df_slovakia.applymap(lambda x: pd.NA if isinstance(x, str) and x.strip().lower() in ['nan', 'none'] else x)


In [163]:
df_slovakia.to_csv("ik_final_upload_ready.csv", index=False)